# DEA Hotspots
Satellite-detected bushfire hotspots across Australia from Geoscience Australia. Updated every 10 minutes. Not for safety-of-life decisions.

## Imports

In [ ]:
import requests
import pandas as pd
import plotly.express as px
from io import StringIO
import warnings, urllib3
warnings.filterwarnings('ignore', category=urllib3.exceptions.NotOpenSSLWarning)

## Latest 72 Hours
From DEA's live GeoJSON feed.

In [ ]:
r = requests.get('https://hotspots.dea.ga.gov.au/data/recent-hotspots.json')
features = r.json()['features']

dea_72h_df = pd.DataFrame([{
    **f['properties'],
    'lon': f['geometry']['coordinates'][0],
    'lat': f['geometry']['coordinates'][1]
} for f in features])

dea_72h_df['temp_celsius'] = dea_72h_df['temp_kelvin'] - 273.15

print(f'Total: {len(dea_72h_df)} hotspots')
dea_72h_df.head()

In [ ]:
fig = px.scatter_map(
    dea_72h_df,
    lat='lat', lon='lon',
    color='temp_celsius',
    size='temp_celsius',
    color_continuous_scale='hot',
    map_style='carto-positron',
    zoom=3, center={'lat': -25.0, 'lon': 134.0},
    hover_data=['datetime', 'sensor', 'confidence', 'australian_state'],
    title='DEA Hotspots - Last 72 Hours'
)
fig.show()

## Last 30 Days
From NASA FIRMS — same underlying VIIRS satellite data, just a longer window. Free, no API key needed for NRT data.

In [ ]:
r = requests.get(
    'https://firms.modaps.eosdis.nasa.gov/api/country/csv/nokey/VIIRS_SNPP_NRT/AUS/30'
)
dea_30d_df = pd.read_csv(StringIO(r.text))
dea_30d_df['temp_celsius'] = dea_30d_df['bright_ti4'] - 273.15

print(f'Total: {len(dea_30d_df)} hotspots')
dea_30d_df.head()

In [ ]:
fig = px.scatter_map(
    dea_30d_df,
    lat='latitude', lon='longitude',
    color='temp_celsius',
    size='temp_celsius',
    color_continuous_scale='hot',
    map_style='carto-positron',
    zoom=3, center={'lat': -25.0, 'lon': 134.0},
    hover_data=['acq_date', 'instrument', 'confidence'],
    title='Australia Hotspots - Last 30 Days (NASA FIRMS)'
)
fig.show()